# Set Up Notebook

In [ ]:
##########
# IMPORT #
##########

# Data processing and math
import pandas as pd
import numpy as np

# Statistics
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display handling
import warnings


#################
# CONFIGURATION #
#################

# Suppress warnings
warnings.filterwarnings('ignore')

# Set global plotting style
sns.set_theme(style = "whitegrid")


#################
# VISUALIZATION #
#################
palette_main = {
    "7-Point":  "#D55E00",
    "13-Point": "#0072B2"
}

# Transform Data

In [ ]:
# Load data
df = pd.read_csv('blame_praise_self_pilot_3_data.csv')

# Define factors
factors_map = {
    'non,blameAc':       ('Non-Ignorant', '7-Point'),
    'ignorant,blame,AC': ('Ignorant',     '7-Point'),
    'non13p':            ('Non-Ignorant', '13-Point'),
    'ignorant13pAC':     ('Ignorant',     '13-Point')
}

# Reshape wide to long
long_data = []

for _, row in df.iterrows():
    p_id = row['ID']
    
    for col, (upbringing, scale) in factors_map.items():
        if col in df.columns and pd.notna(row[col]):
            val = pd.to_numeric(row[col], errors = 'coerce')

            score = val - 7
            
            long_data.append({
                'ID': p_id,
                'Upbringing': upbringing,
                'Scale': scale,
                'Value': val,
                'Score': score,
                'Gender': row.get('Gender'),
                'Age': row.get('Age')
            })

df_long = pd.DataFrame(long_data)
print(f"Transformation complete ({len(df_long)} Observations).")

# Calculate Descriptive Statistics

In [ ]:
# Define group factors and dependent variables
group_factors = ['Upbringing', 'Scale']
dependent_vars = ['Value', 'Score']

# Calculate descriptive statistics
descriptive_stats = df_long.groupby(group_factors)[dependent_vars].agg(['mean', 'std', 'count']).round(3)

# Display results
display(descriptive_stats)

# Perform ANOVAs

In [ ]:
for dv in ['Value', 'Score']:
    print(f"\nANOVA ({dv})")

    # Define formula
    formula = f"{dv} ~ C(Upbringing) * C(Scale)"
    
    # Drop lines with NaN
    df_anova = df_long.dropna(subset=[dv])

    # Fit model
    model = ols(formula, data = df_anova).fit()
    
    # Run ANOVA
    aov_table = sm.stats.anova_lm(model, typ = 2)

    # Calculate effect sizes
    aov_table['partial_eta_sq'] = aov_table['sum_sq'] / (aov_table['sum_sq'] + aov_table.loc['Residual', 'sum_sq'])

    # Display results
    display(aov_table.round(3))

# Generate Bar Plots

In [ ]:
# Define labels and configurations
plot_config = {
    'Value': {
        'title': 'Raw Values (1 to 7 vs. 1 to 13)',
        'ylabel': 'Mean Raw Value',
        'ylim': (0, 14),
        'hline': None
    },
    'Score': {
        'title': 'Transformed Scores (Centered at 0)',
        'ylabel': 'Mean Score (–6 to +6)',
        'ylim': (-7, 7),
        'hline': 0
    }
}

for dv in ['Value', 'Score']:
    config = plot_config[dv]
    
    # Create graphs
    g = sns.catplot(data = df_long,
                    x = "Upbringing",
                    y = dv,
                    hue = "Scale",
                    kind = "bar",
                    palette = palette_main,
                    capsize = .05,
                    height = 5,
                    aspect = 1.2
                   )
    
    # Set axis labels and titles
    g.set_axis_labels("Upbringing", config['ylabel'])
    g.fig.suptitle(config['title'], fontsize = 16)
    
    # Add horizontal zero lines and set y-axis limits
    for ax in g.axes.flat:
        if config['hline'] is not None:
            ax.axhline(config['hline'], color = 'black', lw = 1)
        ax.set_ylim(config['ylim'])
    
    # Make room for title
    plt.subplots_adjust(top = 0.85)
    
    # Export graphs
    filename = f'blame_praise_self_pilot_3_bar_plot_{dv.lower()}.png'
    g.savefig(filename, dpi = 300, bbox_inches = 'tight')
    plt.show()    
    print(f"Graph saved as '{filename}'.")